In [48]:
"""Test functions in a seperate environment
"""
import kpi
import ScheduleChecker as sc
import pandas as pd
from importlib import reload as re
import numpy as np

In [6]:
re(sc)
re(kpi)

<module 'kpi' from 'c:\\Users\\Caspe\\Desktop\\School\\Y2\\Project5\\Project5Group1\\GInasFolder\\kpi.py'>

In [7]:
bp = pd.read_excel('Bus Planning.xlsx')
dm = pd.read_excel('DistanceMatrix.xlsx')
tt = pd.read_excel('Timetable.xlsx')

In [31]:
bp

,start location,end location,start time,end time,activity,line,energy consumption,bus
0,ehvgar,ehvbst,05:03:00,05:07:00,material trip,NaN,1.9800,1
1,ehvbst,ehvapt,05:07:00,05:31:00,service trip,401.0,10.8036,1
2,ehvapt,ehvapt,05:31:00,06:04:00,idle,NaN,5.0000,1
3,ehvapt,ehvbst,06:04:00,06:29:00,service trip,401.0,10.8600,1
4,ehvbst,ehvbst,06:29:00,06:31:00,idle,NaN,5.0000,1
...,...,...,...,...,...,...,...,...
715,ehvbst,ehvapt,17:06:00,17:30:00,service trip,400.0,12.8496,19
716,ehvapt,ehvgar,17:30:00,17:50:00,material trip,NaN,10.8000,19
717,ehvgar,ehvbst,14:02:00,14:06:00,material trip,NaN,1.9800,20
718,ehvbst,ehvapt,14:06:00,14:30:00,service trip,400.0,12.8496,20


In [9]:
dm

,start,end,min_travel_time,max_travel_time,distance_m,line
0,ehvapt,ehvbst,21,23,10250,400.0
1,ehvbst,ehvapt,21,23,10708,400.0
2,ehvapt,ehvbst,22,25,9050,401.0
3,ehvbst,ehvapt,22,24,9003,401.0
4,ehvapt,ehvbst,20,20,8600,NaN
5,ehvbst,ehvapt,20,20,8600,NaN
6,ehvbst,ehvgar,4,4,1650,NaN
7,ehvgar,ehvbst,4,4,1650,NaN
8,ehvapt,ehvgar,20,20,9000,NaN
9,ehvgar,ehvapt,20,20,9000,NaN


In [10]:
tt

,start,departure_time,end,line
0,ehvapt,06:04,ehvbst,401
1,ehvapt,06:34,ehvbst,401
2,ehvapt,06:56,ehvbst,401
3,ehvapt,07:11,ehvbst,401
4,ehvapt,07:26,ehvbst,401
...,...,...,...,...
322,ehvbst,18:52,ehvapt,400
323,ehvbst,18:54,ehvapt,400
324,ehvbst,19:07,ehvapt,400
325,ehvbst,19:22,ehvapt,400


In [105]:
mtt = kpi.merge_route_plan(tt, dm)
mtt['departure_time'] = sc.convert_time_to_minutes(mtt['departure_time'])
mtt['departure_time'] = sc.adjust_for_midnight(mtt['departure_time'])
mtt = mtt.sort_values(['line', 'departure_time']).reset_index(drop=True)
mtt['arrival_time'] = mtt['departure_time'] + mtt['max_travel_time']
mtt['overlap'] = mtt['departure_time'] - mtt.groupby(['line'])['arrival_time'].shift(1)
#AI was asked about how to apply groupby. Tried but failed to apply myself.


In [82]:
print(mtt.loc[
    (mtt['start'] == 'ehvapt') &
    (mtt['line'] == 401)].tail(5)
)
print(mtt.loc[
    (mtt['start'] == 'ehvapt') &
    (mtt['line'] == 400)].head(5)
)

     start  departure_time     end  line  distance_m  min_travel_time  \
82  ehvapt            1355  ehvbst   401        9050               22   
83  ehvapt            1385  ehvbst   401        9050               22   
84  ehvapt            1415  ehvbst   401        9050               22   
85  ehvapt            1441  ehvbst   401        9050               22   
86  ehvapt            1471  ehvbst   401        9050               22   

    max_travel_time  arrival_time  difference  
82               25          1380        55.0  
83               25          1410        55.0  
84               25          1440        55.0  
85               25          1466        51.0  
86               25          1496        55.0  
     start  departure_time     end  line  distance_m  min_travel_time  \
87  ehvapt             439  ehvbst   400       10250               21   
88  ehvapt             454  ehvbst   400       10250               21   
89  ehvapt             469  ehvbst   400       10250  

In [85]:
len(mtt[mtt['overlap'] < 0])

0

In [106]:
mtt

,start,departure_time,end,line,distance_m,min_travel_time,max_travel_time,arrival_time,overlap
0,ehvbst,412,ehvapt,400,10708,21,23,435,NaN
1,ehvbst,426,ehvapt,400,10708,21,23,449,-9.0
2,ehvapt,439,ehvbst,400,10250,21,23,462,-10.0
3,ehvbst,441,ehvapt,400,10708,21,23,464,-21.0
4,ehvapt,454,ehvbst,400,10250,21,23,477,-10.0
...,...,...,...,...,...,...,...,...,...
322,ehvapt,1415,ehvbst,401,9050,22,25,1440,4.0
323,ehvbst,1417,ehvapt,401,9003,22,24,1441,-23.0
324,ehvapt,1441,ehvbst,401,9050,22,25,1466,0.0
325,ehvbst,1449,ehvapt,401,9003,22,24,1473,-17.0


In [ ]:
#New algortim: Take a starting bus and have it take the earliest route.
#Then for each route that starts before that bus, create a new bus to take that route.
#Have each bus (in order of first finished) take the next route until all routes are done. 
#For any route that is missed, add a new bus.